In [1]:
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import Literal
from pathlib import Path
import csv
import json
import os
import re
import subprocess

import tiktoken
from markitdown import MarkItDown
from gitsource import chunk_documents
from minsearch import Index

openai_client = OpenAI()


In [2]:
BOOKS_CSV_URL = "https://raw.githubusercontent.com/alexeygrigorev/ai-engineering-buildcamp-code/main/01-foundation/homework/books.csv"
WORKDIR = Path('.')
BOOKS_DIR = WORKDIR / 'books'
BOOKS_TEXT_DIR = WORKDIR / 'books_text'
CSV_PATH = WORKDIR / 'books.csv'

BOOKS_DIR.mkdir(exist_ok=True)
BOOKS_TEXT_DIR.mkdir(exist_ok=True)

QUESTION = "python function definition"
MODEL = "gpt-4o-mini"


In [3]:
def slugify(value: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", value.lower()).strip("_")


def pick_closest(value: int, options: list[int]) -> int:
    return min(options, key=lambda option: abs(option - value))


In [4]:
if not CSV_PATH.exists():
    subprocess.run(["curl", "-fsSL", BOOKS_CSV_URL, "-o", str(CSV_PATH)], check=True)

with CSV_PATH.open('r', encoding='utf-8', newline='') as f:
    books = list(csv.DictReader(f))

print(f"rows in books.csv: {len(books)}")
books


rows in books.csv: 7


[{'title': 'Think Python 2e',
  'book_url': 'https://greenteapress.com/wp/think-python-2e/',
  'pdf_url': 'http://greenteapress.com/thinkpython2/thinkpython2.pdf'},
 {'title': 'Think DSP',
  'book_url': 'https://greenteapress.com/wp/think-dsp/',
  'pdf_url': 'http://greenteapress.com/thinkdsp/thinkdsp.pdf'},
 {'title': 'Think Complexity 2e',
  'book_url': 'https://greenteapress.com/wp/think-complexity/',
  'pdf_url': 'http://greenteapress.com/complexity2/thinkcomplexity2.pdf'},
 {'title': 'Think Java 2e',
  'book_url': 'https://greenteapress.com/wp/think-java-2e/',
  'pdf_url': 'http://greenteapress.com/thinkjava7/thinkjava2.pdf'},
 {'title': 'Physical Modeling in MATLAB',
  'book_url': 'https://greenteapress.com/wp/physical-modeling-in-matlab/',
  'pdf_url': 'https://github.com/AllenDowney/PhysicalModelingInMatlab/raw/master/PhysicalModelingInMatlab4.pdf'},
 {'title': 'Think OS',
  'book_url': 'https://greenteapress.com/wp/think-os/',
  'pdf_url': 'http://greenteapress.com/thinkos/thi

In [5]:
for row in books:
    title = row['title'].strip()
    pdf_url = row['pdf_url'].strip()
    ext = Path(pdf_url.split('?')[0]).suffix or '.pdf'
    out_path = BOOKS_DIR / f"{slugify(title)}{ext}"

    if out_path.exists() and out_path.stat().st_size > 0:
        continue

    print(f"Downloading {title} -> {out_path.name}")
    subprocess.run([
        "curl", "-fL", "--retry", "3", "--retry-delay", "2", pdf_url, "-o", str(out_path)
    ], check=True)

print(f"downloaded pdf count: {len(list(BOOKS_DIR.glob('*.pdf')))}")


downloaded pdf count: 7


In [6]:
converter = MarkItDown()

for pdf_path in sorted(BOOKS_DIR.glob('*.pdf')):
    out_md = BOOKS_TEXT_DIR / f"{pdf_path.stem}.md"
    if out_md.exists():
        continue

    result = converter.convert(str(pdf_path))
    markdown = result.markdown if result.markdown is not None else result.text_content
    out_md.write_text(markdown, encoding='utf-8')

print(f"markdown file count: {len(list(BOOKS_TEXT_DIR.glob('*.md')))}")


markdown file count: 7


In [7]:
# Question 1: How many lines are in Think Python markdown extraction?
think_python_md = BOOKS_TEXT_DIR / 'think_python_2e.md'
q1_value = think_python_md.read_bytes().count(b'\n')
q1_selected = pick_closest(q1_value, [12268, 14268, 16268, 18268])

print('Q1 raw line count (wc -l):', q1_value)
print('Q1 selected option:', q1_selected)


Q1 raw line count (wc -l): 16268
Q1 selected option: 16268


In [8]:
documents = []

for md_path in sorted(BOOKS_TEXT_DIR.glob('*.md')):
    content = md_path.read_text(encoding='utf-8')
    non_empty_lines = [line for line in content.splitlines() if line.strip()]
    documents.append({'source': md_path.name, 'content': non_empty_lines})

print('documents:', len(documents))


documents: 7


In [9]:
# Question 2: How many chunks for Think Python with size=100 and step=50?
chunks = chunk_documents(documents, size=100, step=50)

chunks_per_source = {}
for chunk in chunks:
    src = chunk['source']
    chunks_per_source[src] = chunks_per_source.get(src, 0) + 1

q2_value = chunks_per_source.get('think_python_2e.md', 0)
q2_selected = pick_closest(q2_value, [134, 214, 294, 374])

print('Q2 raw chunk count (Think Python):', q2_value)
print('Q2 selected option:', q2_selected)


Q2 raw chunk count (Think Python): 214
Q2 selected option: 214


In [10]:
index_docs = []
for i, chunk in enumerate(chunks):
    content = chunk['content']
    content_text = "\n".join(content) if isinstance(content, list) else str(content)
    index_docs.append({
        'id': f'chunk-{i}',
        'source': chunk['source'],
        'start': chunk.get('start', 0),
        'content': content_text,
    })

index = Index(text_fields=['content'], keyword_fields=['id', 'source'])
index.fit(index_docs)

# Question 3: How many documents (chunks) were indexed in total?
q3_value = len(index_docs)
q3_selected = pick_closest(q3_value, [719, 919, 1119, 1319])

print('Q3 raw indexed docs:', q3_value)
print('Q3 selected option:', q3_selected)


Q3 raw indexed docs: 1009
Q3 selected option: 919


In [11]:
# Question 4: Top result source book for query "python function definition"?
results = index.search(QUESTION, num_results=5)
source_to_book = {
    'think_python_2e.md': 'Think Python',
    'think_dsp.md': 'Think DSP',
    'think_java_2e.md': 'Think Java',
    'think_complexity_2e.md': 'Think Complexity',
}

top_source = results[0]['source'] if results else ''
q4_value = source_to_book.get(top_source, top_source)

print('Q4 top source:', top_source)
print('Q4 book:', q4_value)
results[0]


Q4 top source: think_python_2e.md
Q4 book: Think Python


{'id': 'chunk-833',
 'source': 'think_python_2e.md',
 'start': 1900,
 'content': 'when you are comfortable with Python, I’ll make suggestions for installing Python on your\ncomputer.\nThere are a number of web pages you can use to run Python. If you already have a fa-\nvorite, go ahead and use it. Otherwise I recommend PythonAnywhere. I provide detailed\ninstructions for getting started at http://tinyurl.com/thinkpython2e.\nThere are two versions of Python, called Python 2 and Python 3. They are very similar, so\nif you learn one, it is easy to switch to the other. In fact, there are only a few differences you\nwill encounter as a beginner. This book is written for Python 3, but I include some notes\nabout Python 2.\nThe Python interpreter is a program that reads and executes Python code. Depending\non your environment, you might start the interpreter by clicking on an icon, or by typing\npython on a command line. When it starts, you should see output like this:\nPython 3.4.0 (default,

In [12]:
instructions = """
You're a course assistant, your task is to answer the QUESTION from the
course students using the provided CONTEXT
"""

prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()


def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    prompt = prompt_template.format(
        question=question,
        context=context
    ).strip()
    return prompt


def search(question):
    return index.search(question, num_results=5)


def llm(user_prompt, instructions, model='gpt-4o-mini'):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    usage = response.usage
    return {
        'answer': response.output_text,
        'input_tokens': int(getattr(usage, 'input_tokens', 0) or 0),
        'output_tokens': int(getattr(usage, 'output_tokens', 0) or 0),
        'token_source': 'live',
    }


def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)

    if not os.environ.get('OPENAI_API_KEY'):
        enc = tiktoken.encoding_for_model(MODEL)
        estimated_input = 2 + 4 + len(enc.encode('system')) + len(enc.encode(instructions)) + 4 + len(enc.encode('user')) + len(enc.encode(prompt))
        return {
            'answer': 'OPENAI_API_KEY not set. Configure it to run live RAG.',
            'input_tokens': estimated_input,
            'output_tokens': None,
            'token_source': 'estimated',
            'prompt': prompt,
            'search_results': search_results,
        }

    response = llm(prompt, instructions)
    response['prompt'] = prompt
    response['search_results'] = search_results
    return response


In [13]:
# Question 5: Input tokens used for RAG query "python function definition"?
q5_result = rag(QUESTION)
q5_value = int(q5_result['input_tokens'])
q5_selected = pick_closest(q5_value, [4889, 6889, 8889, 10889])

print('Q5 response:')
print(q5_result['answer'])
print('\nQ5 input tokens:', q5_value, f"({q5_result['token_source']})")
print('Q5 output tokens:', q5_result['output_tokens'])
print('Q5 selected option:', q5_selected)


Q5 response:
In Python, a function is defined using the `def` keyword followed by the function name and parentheses containing any parameters. The basics of a function definition include:

1. **Function Definition**: 
   - Start with the `def` keyword.
   - Specify the function name.
   - Include parentheses with optional parameters.
   - End with a colon `:`.

2. **Function Body**: 
   - Indent the following lines for the body of the function.
   - Include statements that define the operations the function performs.

3. **Return Statement** (optional):
   - Use the `return` keyword to output a value from the function, if needed.

### Example:
Here’s a simple example to illustrate the structure of a function:

```python
def greet(name):
    print("Hello, " + name + "!")
```

In this example:
- `greet` is the function name.
- `name` is a parameter.
- The function prints a greeting message.

To call the function, you would use:

```python
greet("Alice")
```

This would output: `Hello, Al

In [14]:
from pydantic import BaseModel, Field, ConfigDict
from typing import Literal

class RAGResponse(BaseModel):
    # Required by Responses API strict json_schema validation
    model_config = ConfigDict(extra='forbid')

    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal['how-to', 'explanation', 'troubleshooting', 'comparison', 'reference'] = Field(description='The category of the answer')
    followup_questions: list[str] = Field(description='Suggested follow-up questions')


def llm_structured(user_prompt, instructions, model='gpt-4o-mini'):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=messages,
        text={
            'format': {
                'type': 'json_schema',
                'name': 'rag_response',
                'schema': RAGResponse.model_json_schema(),
                'strict': True,
            }
        },
    )

    usage = response.usage
    return {
        'response': RAGResponse.model_validate_json(response.output_text).model_dump(),
        'input_tokens': int(getattr(usage, 'input_tokens', 0) or 0),
        'output_tokens': int(getattr(usage, 'output_tokens', 0) or 0),
        'token_source': 'live',
    }


def rag_structured(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)

    if not os.environ.get('OPENAI_API_KEY'):
        enc = tiktoken.encoding_for_model(MODEL)
        base = 2 + 4 + len(enc.encode('system')) + len(enc.encode(instructions)) + 4 + len(enc.encode('user')) + len(enc.encode(prompt))
        schema_tokens = len(enc.encode(json.dumps(RAGResponse.model_json_schema(), indent=2)))
        return {
            'response': {
                'answer': 'OPENAI_API_KEY not set. Configure it to run live structured RAG.',
                'found_answer': False,
                'confidence': 0.0,
                'confidence_explanation': 'No live model call in fallback mode.',
                'answer_type': 'reference',
                'followup_questions': [],
            },
            'input_tokens': base + schema_tokens + 100,
            'output_tokens': None,
            'token_source': 'estimated',
        }

    return llm_structured(prompt, instructions)


In [15]:
# Question 6: Extra input tokens for structured output vs unstructured?
q6_result = rag_structured(QUESTION)
q6_more_tokens = int(q6_result['input_tokens']) - int(q5_result['input_tokens'])
q6_selected = pick_closest(q6_more_tokens, [24, 224, 424, 624])

print('Q6 structured response:')
print(json.dumps(q6_result['response'], indent=2))
print()
print('Q6 structured input tokens:', q6_result['input_tokens'], f"({q6_result['token_source']})")
print('Q5 input tokens:', q5_result['input_tokens'], f"({q5_result['token_source']})")
print('Q6 more input tokens:', q6_more_tokens)
print('Q6 selected option:', q6_selected)


Q6 structured response:
{
  "answer": "### Python Function Definition\n\nIn Python, a function is defined using the `def` statement. Here's the basic syntax:\n\n```python\ndef function_name(parameters):\n    # function body\n    # return statement (optional)\n```\n\n#### Components:\n1. **`def`**: This keyword starts the function definition.\n2. **`function_name`**: This is the name you give to the function. It should be descriptive of what the function does.\n3. **`parameters`**: These are optional inputs that the function can accept, enclosed in parentheses. You can have multiple parameters separated by commas.\n4. **Function Body**: This is the code block that performs the operation of the function. It should be indented.\n5. **Return Statement**: This is optional and specifies the value that the function will return when it is called.\n\n#### Example:\nHere\u2019s an example function that adds two numbers:\n\n```python\ndef add_numbers(a, b):\n    result = a + b\n    return result\

In [16]:
summary = {
    'q1': {'raw': q1_value, 'selected_option': q1_selected},
    'q2': {'raw': q2_value, 'selected_option': q2_selected},
    'q3': {'raw': q3_value, 'selected_option': q3_selected},
    'q4': {'value': q4_value},
    'q5': {'raw_input_tokens': q5_value, 'selected_option': q5_selected, 'token_source': q5_result['token_source']},
    'q6': {'raw_more_input_tokens': q6_more_tokens, 'selected_option': q6_selected, 'token_source': q6_result['token_source']},
}

print(json.dumps(summary, indent=2))


{
  "q1": {
    "raw": 16268,
    "selected_option": 16268
  },
  "q2": {
    "raw": 214,
    "selected_option": 214
  },
  "q3": {
    "raw": 1009,
    "selected_option": 919
  },
  "q4": {
    "value": "Think Python"
  },
  "q5": {
    "raw_input_tokens": 7356,
    "selected_option": 6889,
    "token_source": "live"
  },
  "q6": {
    "raw_more_input_tokens": 185,
    "selected_option": 224,
    "token_source": "live"
  }
}
